[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zshakespear/bd-spectra-onramp/blob/main/notebooks/04_spectral_sequence.ipynb)

# 4 · The spectral sequence

*Brown Dwarf Spectra On-Ramp · about 40 minutes*

**By the end of this notebook you will be able to:**
- describe how brown dwarf spectra change from L0 to ~Y4
- say when CH<sub>4</sub> appears, what happens at the L/T transition, and how late-T and Y dwarfs differ
- estimate a rough spectral type by eye
- appreciate how small the differences between adjacent subtypes are

> **First time here?** Do *File → Save a copy in Drive* before you change anything (notebook 0 explains why).
> Then run the **Setup** cell below. Run cells in order, top to bottom, with **Shift + Enter**.

In [ ]:
# ▶ SETUP - run this cell first. It is safe to run more than once.
REPO_URL = "https://github.com/zshakespear/bd-spectra-onramp.git"
import os, subprocess, sys
from pathlib import Path

if "google.colab" in sys.modules:                       # running in Google Colab
    ROOT = Path("/content/bd-spectra-onramp")
    if not (ROOT / "bdtools").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only", "--quiet"], check=False)
else:                                                   # running in Jupyter on your own computer
    ROOT = Path.cwd()
    while not (ROOT / "bdtools").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import bdtools as bd
bd.setup_notebook()

## 1. The reference sequence

Rustamkulov et al. (2026) assembled SPHEREx spectra of one object per subtype from L0 to ~Y4
(their Table 1; ~2500 K down to ~250 K). These are our **references**.

In [ ]:
cat = bd.list_references()
cat

> The spectra are SPHEREx spectra from the SPIFF library (Gagné et al. 2026), one object per subtype in Table 1.
> Three Table 1 objects are not in the public SPIFF release. For L9 and Y1 we use a **stand-in**: another object
> of the same type (`status` = `substitute`; `table_name` gives the object it replaces). There is no stand-in
> for ~Y4 (WISE 0855−0714, the coldest known brown dwarf), so our sequence stops at Y2.

In [ ]:
# ✏️ SETTINGS YOU CAN CHANGE
NORMALIZE_AT = (1.20, 1.30)   # divide each spectrum by its mean flux here (J peak) to compare shapes
OFFSET = 0.6                  # vertical spacing between stacked spectra

In [ ]:
seq = bd.load_sequence()
print("loaded:", [s.meta["spectral_type"] for s in seq])
fig, ax = bd.plot_spectrum(seq, normalize_at=NORMALIZE_AT, offset=OFFSET, errors=False,
                           colors=list(plt.cm.viridis(np.linspace(0, 0.9, len(seq)))),
                           labels=[s.meta["spectral_type"] for s in seq], title="The sequence (normalized, stacked)")

In [ ]:
fig, ax = bd.plot_spectrum(seq, log=True, normalize_at=NORMALIZE_AT, errors=False,
                           colors=list(plt.cm.viridis(np.linspace(0, 0.9, len(seq)))),
                           labels=[s.meta["spectral_type"] for s in seq], title="The sequence on a log axis")

## 2. What changes along the sequence

These milestones are from Rustamkulov et al. (2026, Sec. 3). Check each one against the plots.

| Where | What to look for |
|---|---|
| **After L4** | CH<sub>4</sub> becomes visible, first as a downward curve at 3–3.6 µm, later dominating the short side of the 4 µm peak. |
| **Beyond L3** | CO<sub>2</sub> (4.15–4.3 µm) and CO (4.4–4.9 µm) notches grow; from L4 a small peak appears at 4.67 µm. |
| **T0** | The K peak (≈2.1 µm) drops below the H peak (≈1.6 µm). |
| **T4** | The H peak drops below the J peak (≈1.27 µm). A narrow CH<sub>4</sub> feature appears at 1.15 µm. |
| **L/T transition** | The 4 µm peak is sharpest here. Clouds clear and the J band brightens. |
| **T6 → T7** (~1000 → 700 K) | The CO and CO<sub>2</sub> indices drop sharply. |
| **Late T and Y** | The 4 µm peak shifts redward. For Y dwarfs, only flux beyond ~3.9 µm is reliably detected. |

### Brightness temperature along the sequence
Converting each spectrum to T<sub>b</sub> (notebook 1) puts all of them on one comparable scale.

In [ ]:
fig, ax = plt.subplots()
colors = plt.cm.viridis(np.linspace(0, 0.9, len(seq)))
for s, c in zip(seq, colors):
    surf = bd.observed_to_surface(s.flux, s.meta["distance_pc"], radius_rjup=1.0)
    ax.plot(s.wave_um, bd.brightness_temperature(s.wave_um, surf), color=c, label=s.meta["spectral_type"])
ax.set_xlabel("Wavelength [µm]"); ax.set_ylabel("brightness temperature [K] (R = 1 R_Jup assumed)")
ax.legend(ncol=4, fontsize=8); ax.set_title("Brightness temperature along the sequence");

## 3. Indices along the sequence

The three indices of Rustamkulov et al. (2026, Table 5) measure band depths in the 3–5 µm range
(details in notebook 6 and the [field guide](../field_guide/indices.md)). Larger = deeper band.

In [ ]:
tab = bd.index_table(seq)
tab

In [ ]:
fig, ax = plt.subplots()
x = np.arange(len(tab))
for name, c in [("CH4", bd.molecule_color("CH4")), ("CO2", bd.molecule_color("CO2")), ("CO", bd.molecule_color("CO"))]:
    ax.errorbar(x, tab[name], yerr=tab[name + "_err"], fmt="o-", color=c, label=name, capsize=2)
ax.set_xticks(x, tab["type"]); ax.set_ylabel("index [mag]  (bigger = deeper band)"); ax.legend()
ax.set_title("Indices along the sequence");

## 4. Estimate a spectral type by eye

Run the next cell to draw a random reference and hide its label. Use the milestones above, then reveal.

In [ ]:
rng = np.random.default_rng()           # put a number inside () to get the same 'mystery' every time
mystery = seq[rng.integers(len(seq))]
fig, ax = bd.plot_spectrum(mystery.copy(name="mystery object"), log=True, title="What type is this?")

In [ ]:
print("It was:", mystery.meta["spectral_type"])

## 5. Adjacent subtypes are subtle

Your real task will often be judging between neighbouring subtypes (an L8 versus an L9 fit, say). Overlay two
neighbours, normalized at the J peak, and look at their ratio.

In [ ]:
# ✏️ SETTINGS YOU CAN CHANGE
PAIR = ("L8", "L9")      # try ("L4", "L5") or ("T6", "T7") too

In [ ]:
a = bd.optional(bd.load_reference, PAIR[0]); b = bd.optional(bd.load_reference, PAIR[1])
if a is not None and b is not None:
    fig, ax = bd.plot_spectrum([a, b], log=True, normalize_at=NORMALIZE_AT, title=f"{PAIR[0]} vs {PAIR[1]}")
    c = bd.compare(a, b, windows=[NORMALIZE_AT])
    fig, axes = bd.plot_comparison(c, panel="both", title=f"{PAIR[0]} / {PAIR[1]} (scaled at the J peak)")

Most of the ratio is within a few percent of 1, and the largest differences sit in specific bands. That is
normal — and it is why the comparison notebook (6) is about uncertainties and significance, not just shapes.

### Exercise 4.1
On the stacked plot, at which subtype does a downward curve at 3–3.6 µm first become obvious?

<details><summary><b>Answer</b> (try first, then click)</summary>

Rustamkulov et al. (2026) find CH<sub>4</sub> becomes visible after L4. With noisy data you may not see it
clearly until L5–L6; that is fine — say what *you* see and how confident you are.

</details>

### Exercise 4.2
Using the index plot, which index changes most steadily across the L sequence?

<details><summary><b>Answer</b> (try first, then click)</summary>

Read it off your plot. Expect the CO and CO<sub>2</sub> indices to grow beyond L3 and the CH<sub>4</sub> index to rise
after L4 (Rustamkulov et al. 2026). Check the error bars before calling a trend real.

</details>

## Summary

- Going later: the flux peak moves to longer wavelengths and CH<sub>4</sub>, CO and CO<sub>2</sub> features develop in a known order.
- Key markers: CH<sub>4</sub> after L4; K < H at T0; H < J at T4; CO/CO<sub>2</sub> collapse between T6 and T7; Y dwarfs are detected only beyond ~3.9 µm.
- Adjacent subtypes differ by a few percent — judge with uncertainties.
- Field guide: [Spectral sequence cheat sheet](../field_guide/spectral_sequence.md)

**Field guide:** the pages linked above are the short versions to keep open while you work.

## Self-check

Answer these without scrolling up. If one is hard, revisit that section.

1. At which subtype does the K peak drop below the H peak?
<details><summary><b>Answer</b> (try first, then click)</summary>

T0 (Rustamkulov et al. 2026).

</details>

2. What happens to the CO and CO<sub>2</sub> indices between T6 and T7?
<details><summary><b>Answer</b> (try first, then click)</summary>

They drop sharply (~1000 → 700 K).

</details>

3. Where is the 4 µm peak sharpest?
<details><summary><b>Answer</b> (try first, then click)</summary>

At the L/T transition.

</details>

4. Why do Y-dwarf spectra look empty below ~3.9 µm?
<details><summary><b>Answer</b> (try first, then click)</summary>

Too little flux there to be reliably detected by SPHEREx.

</details>

**Next:** [Notebook 5 · Global atmospheric parameters](05_global_parameters.ipynb)